# How the Ticket Classification Model Works

This notebook walks through the **machine-learning (ML) classifier** used by the Week 5 support-ticket project. It follows the real preprocessing and pipeline builders in `scripts/preprocess_data.py` and `scripts/train_models.py` rather than inventing a separate model implementation.

The walkthrough covers the data, TF-IDF text features, candidate classifiers, validation, model selection, confidence scores, and how the API loads a saved model. It does not call Gemini or send ticket text to an external service.

## Before you begin

- Open this notebook with the Python environment that has the project's dependencies installed. In VS Code, choose the project's `.venv` kernel.
- Run cells from top to bottom. The notebook locates the project root whether the current directory is `Week-5/Project` or its `notebook` subfolder.
- This walkthrough uses `data/raw/demo_tickets.csv`, a small **synthetic example**. Its validation scores are for learning the workflow only; they are not evidence of production accuracy.
- The normalization example writes to a temporary directory, so it does not overwrite project data. Model comparisons and predictions are kept in memory; the integration example also writes its temporary model artifact outside the project.

The deployed app also compares against Gemini and a Random Forest model. This notebook focuses on understanding the project's TF-IDF candidate models and their API integration.

In [ ]:
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

# Find the project root from this notebook's working directory.
ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "scripts" / "train_models.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find scripts/train_models.py. Open this notebook inside Week-5/Project."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.core.categories import CATEGORY_NAMES
from app.services.ml_classifier import MLClassifier
from scripts.preprocess_data import normalize_dataset
from scripts.train_models import build_candidates, build_random_forest_pipeline

print(f"Project root: {ROOT}")

## 1. Prepare and inspect the data

Each training row needs ticket text and one supported category. The project's normalizer accepts common source column names, canonicalizes category spelling, removes empty/duplicate rows, and rejects labels outside the project's fixed taxonomy. If a source has a `subject`, it is combined with the description.

In [ ]:
raw_path = ROOT / "data" / "raw" / "demo_tickets.csv"
raw_tickets = pd.read_csv(raw_path)
print(f"Raw rows and columns: {raw_tickets.shape}")
display(raw_tickets.head())

# Reuse the project's actual preprocessing function without changing project files.
with TemporaryDirectory(prefix="ticket-notebook-") as temporary_directory:
    normalized_path = Path(temporary_directory) / "tickets.csv"
    tickets = normalize_dataset(raw_path, normalized_path)

print(f"Normalized rows and columns: {tickets.shape}")
display(tickets.head())

In [ ]:
category_counts = tickets["category"].value_counts().reindex(CATEGORY_NAMES, fill_value=0)
display(category_counts.rename("ticket_count").to_frame())
print(f"Number of rows: {len(tickets)}")
print(f"Number of supported categories represented: {tickets['category'].nunique()} / {len(CATEGORY_NAMES)}")
print(f"Smallest category count: {category_counts.min()}")

if (category_counts == 0).any():
    raise ValueError("Training data must include every supported category.")
if category_counts.min() < 4:
    raise ValueError("The project trainer requires at least four tickets per category.")

category_counts.plot(kind="bar", figsize=(12, 4), title="Synthetic demo tickets by category")
plt.ylabel("Tickets")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 2. Hold out validation examples

The project reserves 25% of the rows for validation and uses `stratify=y` so each category is represented in both partitions. `random_state=42` makes the split repeatable. Validation examples are not used to fit a candidate pipeline; they help compare candidate predictions.

In [ ]:
X = tickets["ticket_text"]
y = tickets["category"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Validation rows: {len(X_valid)}")
display(
    pd.DataFrame(
        {
            "training": y_train.value_counts().reindex(CATEGORY_NAMES, fill_value=0),
            "validation": y_valid.value_counts().reindex(CATEGORY_NAMES, fill_value=0),
        }
    )
)

## 3. Turn ticket text into numeric features with TF-IDF

A scikit-learn classifier needs numeric input. `TfidfVectorizer` tokenizes the text and weights terms by how informative they are across the training set. The project uses lowercase text, single words and two-word phrases (`ngram_range=(1, 2)`), and sublinear term frequency scaling. The vectorizer is part of each model pipeline, so it is fit only on training text during validation.

In [ ]:
vectorizer_demo = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)
X_train_tfidf = vectorizer_demo.fit_transform(X_train)
feature_names = vectorizer_demo.get_feature_names_out()
density = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print(f"TF-IDF matrix shape (tickets, features): {X_train_tfidf.shape}")
print(f"Non-zero feature density: {density:.3%}")

# Inspect the highest-weight terms for one training ticket.
first_ticket_weights = X_train_tfidf[0].toarray().ravel()
display(
    pd.Series(first_ticket_weights, index=feature_names, name="tfidf_weight")
    .nlargest(12)
    .to_frame()
)

## 4. Build the candidate pipelines

The project compares three TF-IDF pipelines:

1. **Logistic Regression** — a linear classifier with balanced class weights.
2. **Linear SVM** — a linear support-vector classifier, wrapped in sigmoid probability calibration.
3. **Multinomial Naive Bayes** — a probabilistic text classifier, also wrapped in sigmoid calibration.

Each pipeline combines TF-IDF and its classifier, preventing the validation text from leaking into vectorizer fitting. `CalibratedClassifierCV(method="sigmoid", cv=3)` supplies probability estimates for the three candidates. A separate Random Forest pipeline is evaluated for comparison, but the project's selected model is the highest-macro-F1 model among the three calibrated candidates.

In [ ]:
candidate_pipelines = build_candidates()
for model_name, pipeline in candidate_pipelines.items():
    print(f"\n{model_name}")
    print(pipeline)

random_forest_pipeline = build_random_forest_pipeline()
print("\nSeparate comparison pipeline:")
print(random_forest_pipeline)

## 5. Train and compare models

Macro F1 is the selection metric: it calculates F1 per category and gives each category equal weight, which is useful when class counts differ. Accuracy, macro precision, and macro recall are shown as additional context. The separate Random Forest is scored on the same held-out split.

In [ ]:
validation_predictions = {}
candidate_scores = {}

for model_name, pipeline in candidate_pipelines.items():
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_valid)
    validation_predictions[model_name] = prediction
    candidate_scores[model_name] = {
        "accuracy": accuracy_score(y_valid, prediction),
        "precision_macro": precision_score(
            y_valid, prediction, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_valid, prediction, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_valid, prediction, average="macro", zero_division=0
        ),
    }

random_forest_pipeline.fit(X_train, y_train)
random_forest_prediction = random_forest_pipeline.predict(X_valid)
validation_predictions["Random Forest"] = random_forest_prediction
candidate_scores["Random Forest"] = {
    "accuracy": accuracy_score(y_valid, random_forest_prediction),
    "precision_macro": precision_score(
        y_valid, random_forest_prediction, average="macro", zero_division=0
    ),
    "recall_macro": recall_score(
        y_valid, random_forest_prediction, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        y_valid, random_forest_prediction, average="macro", zero_division=0
    ),
}

comparison = pd.DataFrame(candidate_scores).T.sort_values("macro_f1", ascending=False)
display(comparison.style.format("{:.3f}"))

# Match the project trainer: select among the three calibrated candidates, not Random Forest.
best_ml_name = max(
    candidate_pipelines,
    key=lambda name: candidate_scores[name]["macro_f1"],
)
print(f"Selected calibrated candidate: {best_ml_name}")
print(f"Validation macro F1: {candidate_scores[best_ml_name]['macro_f1']:.3f}")

### Read the detailed validation results

The report below gives precision, recall, and F1 for each category. With a tiny validation set, a single prediction can change a category's score substantially. Always interpret these demo metrics cautiously.

In [ ]:
best_valid_prediction = validation_predictions[best_ml_name]
print(
    classification_report(
        y_valid,
        best_valid_prediction,
        labels=CATEGORY_NAMES,
        zero_division=0,
    )
)

In [ ]:
matrix = confusion_matrix(y_valid, best_valid_prediction, labels=CATEGORY_NAMES)
display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=CATEGORY_NAMES,
)
figure, axis = plt.subplots(figsize=(12, 9))
display_matrix.plot(
    ax=axis,
    xticks_rotation=45,
    cmap="Blues",
    colorbar=False,
)
axis.set_title(f"Validation confusion matrix: {best_ml_name}")
plt.tight_layout()
plt.show()

## 6. Refit the selected model and classify a new ticket

After comparing candidates, the production training script fits the selected pipeline again on **all available training rows** before saving it. The next cell follows that step in memory, then ranks the calibrated class probabilities for an example ticket. The top probability becomes the ML prediction's confidence value; it is a model estimate, not a guarantee of correctness.

In [ ]:
final_pipeline = candidate_pipelines[best_ml_name]
final_pipeline.fit(X, y)

example_ticket = "I cannot sign in after resetting my password."
probabilities = final_pipeline.predict_proba([example_ticket])[0]
class_probabilities = (
    pd.Series(probabilities, index=final_pipeline.classes_, name="probability")
    .sort_values(ascending=False)
)
display(class_probabilities.to_frame())
print(f"Predicted category: {class_probabilities.index[0]}")
print(f"Top probability: {class_probabilities.iloc[0]:.1%}")

## 7. How the application uses the trained pipeline

The API's `MLClassifier` loads a Joblib artifact containing a fitted `pipeline` and `model_name`. For a ticket, it calls `predict_proba`, finds the largest probability, converts its class to the project's validated category type, and returns the category, confidence, model name, and a short explanation.

This example saves a temporary artifact and passes it through the real loader and prediction service. It does **not** overwrite the application's model files.

In [ ]:
with TemporaryDirectory(prefix="ticket-model-artifact-") as temporary_directory:
    artifact_path = Path(temporary_directory) / "ticket_classifier.joblib"
    joblib.dump(
        {
            "pipeline": final_pipeline,
            "model_name": best_ml_name,
            "categories": CATEGORY_NAMES,
        },
        artifact_path,
    )

    application_classifier = MLClassifier(artifact_path)
    application_classifier.load()
    application_prediction = application_classifier.predict(example_ticket)
    display(application_prediction.model_dump(mode="json"))

## 8. Where the production training outputs go

The notebook kept its candidates and final pipeline in memory. To run the project's full training workflow from `Week-5/Project`, use:

```powershell
python scripts/preprocess_data.py
python scripts/train_models.py
```

The trainer compares Logistic Regression, Linear SVM, and Multinomial Naive Bayes using validation macro F1, evaluates a separate Random Forest, refits the selected calibrated candidate and Random Forest on all rows, and writes model artifacts to `models/`. It also writes ML metrics and a confusion-matrix image under `evaluation/outputs/`. Running the trainer replaces those generated artifacts.

The bundled dataset is intentionally small and synthetic. For a meaningful evaluation, use a representative, properly licensed dataset with enough examples per category and maintain a separate test set for final evaluation.

## Key takeaways

- Input text is normalized, transformed into TF-IDF features, and passed to a scikit-learn text-classification pipeline.
- Three calibrated candidates are compared; **macro F1** determines which candidate the project saves. Random Forest is an additional comparison model.
- The selected pipeline is refit on all training rows and provides category probabilities used by the API for its ML confidence.
- The local demo data is useful for exercising code, not for estimating real-world model quality.

Try changing `example_ticket` and observing the probability table. Then inspect `scripts/train_models.py` to see the source-of-truth implementation used by the project.